# Modelo de precio de departamentos usados en Providencia

**Objetivo:** predecir el precio en UF a partir de dormitorios, baños, superficie útil, latitud y longitud.

El notebook usa el paquete `modelo` (carpeta `src/modelo`). Cada paso llama a una función del paquete, así el notebook y la CLI (interfaz de línea de comandos) ejecutan el mismo código.

In [ ]:
import warnings

import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

from modelo import data, evaluate

warnings.filterwarnings("ignore")
X, y, g = data.load()
print(f"{len(X)} avisos, {g.nunique()} grupos de coordenadas")
X.describe().round(3)

## 1. Distribución espacial del precio

El precio por m² depende de la ubicación. Por eso latitud y longitud son variables del modelo.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(X["lon"], X["lat"], c=y / X["superficie_m2"], cmap="viridis", s=10)
fig.colorbar(sc, label="UF por m²")
ax.set(xlabel="lon", ylabel="lat", title="Precio por m² en Providencia");

**Cómo leer el mapa.** Cada punto es un departamento. La posición horizontal es la longitud: la izquierda es el oeste y la derecha es el este. La posición vertical es la latitud: abajo es el sur y arriba es el norte. El color es el precio por metro cuadrado, en UF (Unidad de Fomento). El amarillo es más caro por m². El morado es más barato. Si el color cambia de un lado del mapa al otro, la ubicación está ligada al precio. Por eso el modelo usa latitud y longitud.


## 2. Partición train y test por grupos

Muchos avisos comparten coordenadas: son el mismo edificio, o la misma calle cuando la geocodificación no encontró el número. Si un edificio aparece en train y en test, el modelo "memoriza" su precio y la métrica es optimista.

Por eso la partición usa **grupos**: cada grupo de coordenadas queda completo en train o en test. El test (20 %) no se usa hasta el final.

In [ ]:
Xtr, Xte, ytr, yte, gtr = evaluate.split(X, y, g)
print(f"train {len(Xtr)} / test {len(Xte)}")

## 3. Ingeniería de variables de ubicación

La latitud casi no se correlaciona con el precio por m². Esa correlación no dice si la columna aporta. Hay que medir el MAE (error absoluto medio) con la columna y sin ella.

La zona es una etiqueta de KMeans (k-medias) sobre latitud y longitud escaladas, con seis grupos. En la tabla, cada pliegue vuelve a ajustar la zona solo con su train. El mapa de la derecha la ajusta con todo el train, para ver los grupos.

El CatBoost de esta tabla tiene hiperparámetros fijos. Su MAE no es el de la validación anidada de la sección siguiente. Sirve para comparar variantes entre sí.

Quitar la latitud sube el MAE. Quitar la longitud lo sube más. Agregar la zona, o usarla en lugar de las coordenadas, no baja el error. Un árbol ya corta el mapa con latitud y longitud, y seis zonas son un resumen más grueso. El modelo se queda con las coordenadas.

In [ ]:
from modelo import features

ppm = ytr / Xtr["superficie_m2"]
print("correlación con UF/m²")
print(Xtr[["lat", "lon"]].corrwith(ppm).round(3).to_string())
tabla = features.comparar(Xtr, ytr, gtr)
tabla.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].barh(tabla.variante, tabla.delta_mae, xerr=tabla.mae_std)
axes[0].axvline(0, color="k", lw=1)
axes[0].set(xlabel="cambio del MAE (UF)", title="Positivo: la variante es peor")
axes[1].scatter(Xtr["lon"], Xtr["lat"], c=features.zonas(Xtr), cmap="tab10", s=10)
axes[1].set(xlabel="lon", ylabel="lat", title="Seis zonas, no entran al modelo")
fig.tight_layout()

**Cómo leer estos dos gráficos.** A la izquierda, la línea vertical es el modelo con latitud y longitud. Cada barra es otra forma de usar la ubicación. Una barra hacia la derecha se equivoca más. El número son UF de error extra. La línea negra muestra cuánto cambia ese resultado entre las cinco pruebas. Si cruza el cero, la diferencia no es clara.

`sin_lat` quita la latitud. `sin_lon` quita la longitud. `sin_coordenadas` quita las dos. `zona` agrega una etiqueta de sector. `solo_zona` reemplaza las coordenadas por esa etiqueta. Las barras largas hacia la derecha son las que se quedan sin coordenadas: la ubicación sí aporta. `zona` queda sobre el cero: seis sectores no mejoran el error.

A la derecha, cada color es uno de esos seis sectores, es decir, departamentos cercanos entre sí. El modelo no usa esos colores, porque no bajan el error.


## 4. Validación cruzada anidada

Se comparan 7 regresores. Todos aprenden `log(precio)` y predicen el precio en UF; el logaritmo corrige la cola larga del precio. `dummy` (mediana) es la línea base: un modelo útil debe superarla.

| Modelo | Tipo |
|---|---|
| dummy | Línea base (mediana) |
| ridge | Lineal con regularización L2 |
| knn | k vecinos más cercanos |
| random_forest | Bosque aleatorio |
| hist_gb | Gradient boosting de scikit-learn |
| xgboost | Gradient boosting (XGBoost) |
| catboost | Gradient boosting (CatBoost) |

La validación cruzada **anidada** tiene dos lazos, ambos con `GroupKFold`:

1. Lazo interno (3 pliegues): `RandomizedSearchCV` prueba 20 combinaciones de hiperparámetros y elige la de menor MAE (error absoluto medio).
2. Lazo externo (5 pliegues): mide el error del modelo ya ajustado en datos que la búsqueda no vio.

Si se usa el mismo lazo para elegir hiperparámetros y para medir el error, la métrica sale optimista. El lazo externo evita ese sesgo. Tiempo aproximado: 1 minuto.

In [ ]:
cv = evaluate.nested_cv(Xtr, ytr, gtr)
res = evaluate.summary(cv)
res.round(3)

In [ ]:
orden = res.index.tolist()
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([cv.loc[cv.modelo == m, "mae"] for m in orden], tick_labels=orden)
ax.set(ylabel="MAE (UF)", title="MAE por pliegue externo (menor es mejor)");

**Cómo leer el gráfico.** Cada caja es un modelo. El eje vertical es el MAE (error absoluto medio): el promedio de cuántas UF se aleja la predicción del precio publicado. Más abajo es mejor.

Cada caja resume cinco pruebas. La línea del centro es la prueba del medio. La caja cubre la mitad central de las pruebas. Las líneas de arriba y abajo llegan a la mejor y a la peor. Una caja alta significa que el error cambia mucho según la prueba.

`dummy` siempre responde el precio mediano. Es la referencia: un modelo útil queda más abajo. El de más a la izquierda es el que menos se equivoca.


## 5. Modelo final y evaluación en test

El modelo con menor MAE medio se vuelve a ajustar con todo el train (búsqueda de hiperparámetros incluida). Luego se evalúa **una sola vez** en el test.

In [ ]:
best = res.index[0]
model = evaluate.fit_final(best, Xtr, ytr, gtr)
pred = model.predict(Xte)
print(best)
{k: round(v, 3) for k, v in evaluate.metrics(yte, pred).items()}

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(yte, pred, s=10, alpha=0.6)
lim = [min(yte.min(), pred.min()), max(yte.max(), pred.max())]
ax.plot(lim, lim, "k--", lw=1)
ax.set(xlabel="precio real (UF)", ylabel="precio predicho (UF)", title=f"{best} en test");

**Cómo leer el gráfico.** Cada punto es un departamento del test, es decir, un aviso que el modelo no usó para aprender. El eje horizontal es el precio publicado, en UF. El vertical es el precio que predice el modelo. La línea punteada es el acierto perfecto: la predicción iguala al precio real.

Un punto sobre la línea es un departamento que el modelo cree más caro de lo publicado. Un punto bajo la línea es uno que cree más barato. Si los puntos acompañan la línea, el modelo distingue un departamento barato de uno caro. Hacia la derecha la nube se abre: en los departamentos caros, el error en UF es más grande.


## 6. Importancia de variables

El gráfico mide cuánto empeora la predicción si se pierde la información de una sola variable. Se mezcla esa columna al azar en el test y se dejan las otras igual. La barra es el error extra, en UF.


In [ ]:
imp = permutation_importance(
    model, Xte, yte, scoring="neg_mean_absolute_error", n_repeats=20, random_state=42
)
orden = sorted(range(X.shape[1]), key=lambda i: imp.importances_mean[i])
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(X.columns[orden], imp.importances_mean[orden], xerr=imp.importances_std[orden])
ax.set(xlabel="aumento del MAE (UF)", title="Importancia por permutación")
fig.tight_layout()

**Cómo leer el gráfico.** Cada barra es una variable. El largo es cuántas UF extra se equivoca el modelo, en el test, si esa variable se mezcla al azar y las demás se dejan quietas. Arriba está la que más duele perder. La línea negra es la variación de repetir la mezcla 20 veces.

De arriba abajo. La superficie útil es la que más importa: cerca de 1.100 UF. La longitud sigue, cerca de 300 UF. Baños, cerca de 250 UF. Dormitorios, cerca de 190 UF. La latitud es la más corta, cerca de 130 UF. Sigue en el modelo, porque quitarla empeora la predicción, aunque menos que quitar la longitud.


## 7. Predicción de un departamento nuevo

In [ ]:
uf = evaluate.predict_one(
    model, dormitorios=2, banos=2, superficie_m2=80, lat=-33.4263, lon=-70.6110
)
print(f"2D/2B, 80 m², cerca de metro Pedro de Valdivia: {uf:,.0f} UF")

## 8. Intervalo con el mismo CatBoost

El modelo de la sección 5 entrega un solo precio. La predicción conformal le agrega un intervalo. El modelo sigue siendo CatBoost. El intervalo sale de los residuos en logaritmo, medidos en avisos que ese CatBoost no usó para ajustarse.

El puntaje de cada aviso de calibración es `s = |log(precio) - log(predicho)|`. `q` es un cuantil alto de esos puntajes. Con alfa = 0,10 el objetivo es cubrir el 90 % de los precios de publicación. Para un departamento nuevo el intervalo es `[predicho · exp(-q), predicho · exp(q)]`. El ancho en UF (Unidad de Fomento) crece con el precio predicho. En proporción, el margen es el mismo.

Los grupos de coordenadas quedan en tres bloques distintos. El test no cambia. Del train, un 20 % de los grupos solo calibra `q`. El resto ajusta el CatBoost, con la misma búsqueda de hiperparámetros. Este CatBoost no es el de la sección 5, porque no vio la calibración. El precio puntual puede diferir.

El ajuste tarda lo mismo que el CatBoost de la sección 5.

In [ ]:
from modelo import conformal

Xfit, Xcal, yfit, ycal, gfit = conformal.split_calibration(Xtr, ytr, gtr)
cat = evaluate.fit_final("catboost", Xfit, yfit, gfit)
q = conformal.calibrate(cat, Xcal, ycal, alpha=0.1)
pred_c = cat.predict(Xte)
print(f"ajuste {len(Xfit)} / calibración {len(Xcal)}")
print(f"q = {q:.3f}. Margen ±{conformal.margin(q):.1%} del precio predicho.")
print("test", {k: round(v, 3) for k, v in evaluate.metrics(yte, pred_c).items()})
print(conformal.coverage(yte, pred_c, q))
tramos = conformal.coverage_by_band(yte, pred_c, q)
tramos.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(tramos.tramo.astype(str), tramos.cobertura)
ax.axhline(0.9, color="k", lw=1)
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=15)
ax.set(ylabel="cobertura", title="Cobertura por tramo de precio predicho")
fig.tight_layout()

**Cómo leer el gráfico.** Cada barra es un tramo de precio predicho. La altura es la fracción de avisos del test cuyo precio publicado cae dentro del intervalo. La línea negra es el 90 %. Una barra bajo la línea es un tramo donde el intervalo se queda corto.

El 90 % es un objetivo sobre todos los avisos juntos. Un tramo puede quedar debajo. La cobertura que imprime la celda de arriba es la de esta partición. También puede quedar bajo el 90 %. El cuantil se calcula en la calibración. El test no lo corrige. El ancho en UF es mayor en el tramo caro, porque el margen es un porcentaje del precio predicho.

In [ ]:
punto, lo, hi = conformal.predict_interval(
    cat, q, dormitorios=2, banos=2, superficie_m2=80, lat=-33.4263, lon=-70.6110
)
print(f"2D/2B, 80 m², cerca de metro Pedro de Valdivia: {punto:,.0f} UF [{lo:,.0f}, {hi:,.0f}]")

## Limitaciones

1. El precio es el de publicación, no el de venta. El intervalo también cubre el precio de publicación.
2. Faltan variables relevantes: antigüedad, piso, estacionamiento, bodega y gastos comunes.
3. Una parte de las coordenadas es aproximada (nivel calle o estación de metro).
4. La cobertura del 90 % es un promedio sobre todos los avisos. Dentro de un tramo de precio, o dentro de un edificio, la cobertura puede ser otra.